# Landslide Susceptibility Prediction using Logistic Regression
### Study Area: Himachal Pradesh, India

This notebook implements end-to-end training, evaluation, and feature interpretability for a **Logistic Regression** classifier with **L2 Regularization**.

The model is evaluated using **12 comprehensive performance metrics**:
1. Accuracy
2. Precision
3. Recall (Sensitivity)
4. Specificity
5. F1-score
6. ROC-AUC (Area Under ROC Curve)
7. PR-AUC (Precision–Recall Area / Average Precision)
8. Balanced Accuracy
9. Matthews Correlation Coefficient (MCC)
10. Cohen's Kappa
11. Confusion Matrix
12. Classification Report

In [ ]:
# 1. Import Required Libraries
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

# Visual theme settings
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries successfully imported!")

## 2. Load the Preprocessed and Engineered Datasets

In [ ]:
# Load Training and Testing Datasets
train_df = pd.read_csv("landslide_train_engineered.csv")
test_df = pd.read_csv("landslide_test_engineered.csv")

# Separate Features (X) and Target (y)
X_train = train_df.drop(columns=["class"])
y_train = train_df["class"]

X_test = test_df.drop(columns=["class"])
y_test = test_df["class"]

print(f"Training Set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing Set:  {X_test.shape[0]} samples, {X_test.shape[1]} features")
print("\nClass Distribution (Train):")
print(y_train.value_counts(normalize=True).rename({0: 'Non-Landslide (0)', 1: 'Landslide (1)'}))
print("\nClass Distribution (Test):")
print(y_test.value_counts(normalize=True).rename({0: 'Non-Landslide (0)', 1: 'Landslide (1)'}))

## 3. Evaluation Function for All 12 Metrics

In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_prob):
    """Calculates and prints all 12 performance evaluation metrics."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    
    print(f"="*65)
    print(f" EVALUATION RESULTS: {model_name}")
    print(f"="*65)
    print(f"1. Accuracy:                                {acc:.4f} ({acc*100:.2f}%)")
    print(f"2. Precision:                               {prec:.4f} ({prec*100:.2f}%)")
    print(f"3. Recall (Sensitivity / True Positive Rate):{rec:.4f} ({rec*100:.2f}%)")
    print(f"4. Specificity (True Negative Rate):        {spec:.4f} ({spec*100:.2f}%)")
    print(f"5. F1-score:                                {f1:.4f} ({f1*100:.2f}%)")
    print(f"6. ROC-AUC:                                 {roc_auc:.4f}")
    print(f"7. PR-AUC (Average Precision):              {pr_auc:.4f}")
    print(f"8. Balanced Accuracy:                       {bal_acc:.4f} ({bal_acc*100:.2f}%)")
    print(f"9. Matthews Correlation Coefficient (MCC):  {mcc:.4f}")
    print(f"10. Cohen's Kappa:                          {kappa:.4f}")
    print(f"\n11. Confusion Matrix:")
    print(f"    [TN: {tn:4d} | FP: {fp:4d}]")
    print(f"    [FN: {fn:4d} | TP: {tp:4d}]")
    print(f"\n12. Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    print(f"="*65 + "\n")
    
    return {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": spec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": bal_acc,
        "MCC": mcc,
        "Cohen's Kappa": kappa,
        "Confusion Matrix": cm
    }

## 4. Build and Train Logistic Regression Pipeline
We package `StandardScaler` and `LogisticRegression(C=0.5, max_iter=1000)` into an end-to-end `Pipeline`.

In [ ]:
# Construct Pipeline
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=0.5, max_iter=1000, random_state=42))
])

print("Training Logistic Regression model...", flush=True)
lr_pipeline.fit(X_train, y_train)
print("Logistic Regression Training successfully completed!")

# Predict on Test Set
y_pred = lr_pipeline.predict(X_test)
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]
decision_scores = lr_pipeline.decision_function(X_test)

# Compute and display all 12 metrics
lr_results = evaluate_model("Logistic Regression (C=0.5)", y_test, y_pred, y_prob)

## 5. Visualizations for Logistic Regression
We generate plots for the Confusion Matrix, ROC/PR Curves, and Feature Coefficients.

In [ ]:
# 1. Confusion Matrix Plot
plt.figure(figsize=(7, 6))
sns.heatmap(
    lr_results["Confusion Matrix"],
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False,
    xticklabels=['Non-Landslide (0)', 'Landslide (1)'],
    yticklabels=['Non-Landslide (0)', 'Landslide (1)'],
    annot_kws={"size": 16, "weight": "bold"}
)
plt.title("Logistic Regression Confusion Matrix", fontsize=14, weight='bold', pad=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Combined ROC and Precision-Recall Curves
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec_pts, rec_pts, _ = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2ca02c', lw=2.5, label=f"LR ROC (AUC = {lr_results['ROC-AUC']:.4f})")
axes[0].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label="Chance Level (0.5000)")
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.02])
axes[0].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
axes[0].set_ylabel("True Positive Rate (Recall / Sensitivity)", fontsize=12)
axes[0].set_title("Logistic Regression ROC Curve", fontsize=13, weight='bold')
axes[0].legend(loc="lower right", frameon=True)
axes[0].grid(True, linestyle=':', alpha=0.6)

# PR Curve
axes[1].plot(rec_pts, prec_pts, color='#d62728', lw=2.5, label=f"LR PR (AP = {lr_results['PR-AUC']:.4f})")
axes[1].axhline(y=y_test.mean(), color='gray', lw=1.5, linestyle='--', label=f"Baseline ({y_test.mean():.4f})")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.02])
axes[1].set_xlabel("Recall (Sensitivity)", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Logistic Regression Precision–Recall Curve", fontsize=13, weight='bold')
axes[1].legend(loc="lower left", frameon=True)
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# 3. Feature Importance (Coefficients)
coefs = lr_pipeline.named_steps['lr'].coef_[0]
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": coefs
}).sort_values(by="Coefficient")

top_features = pd.concat([coef_df.head(8), coef_df.tail(8)]).sort_values(by="Coefficient")

plt.figure(figsize=(10, 6.5))
colors = ['#d62728' if c < 0 else '#1f77b4' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors, height=0.65)
plt.axvline(0, color='black', lw=1.2, linestyle='--')
plt.title("Top Standardized Feature Impacts on Landslide Log-Odds", fontsize=13, weight='bold', pad=12)
plt.xlabel("Standardized Coefficient (Positive = Increases Risk, Negative = Decreases Risk)", fontsize=11)
plt.grid(True, axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Save Model & Export Predictions for Dashboard

In [ ]:
# 1. Save trained model pipeline
joblib.dump(lr_pipeline, "logistic_regression_landslide_model.joblib")
print("Model saved to: logistic_regression_landslide_model.joblib")

# 2. Export predictions table
pred_df = pd.DataFrame({
    "actual_class": y_test.values,
    "predicted_class": y_pred,
    "prob_landslide (class=1)": np.round(y_prob, 4),
    "prob_non_landslide (class=0)": np.round(1 - y_prob, 4),
    "logistic_decision_score": np.round(decision_scores, 4)
})
for kf in ['Slope_z', 'Rainfall_z', 'Elevation_z', 'NDVI_z', 'Dist_River_z']:
    if kf in X_test.columns:
        pred_df[kf] = X_test[kf].values

pred_df.to_csv("logistic_regression_test_predictions.csv", index=True, index_label="sample_id")
print("Predictions saved to: logistic_regression_test_predictions.csv")
print("\nAll Logistic Regression workflow steps completed successfully!")